In [1]:
"""
结构感知修正器训练脚本

核心思路：
1. 让模型学习"产品类型→典型结构→位置修正"的知识链条
2. 不依赖位置先验，而是基于语义结构进行修正决策
3. 使用多任务学习同时优化：结构预测、字符修正、编辑决策
"""

"""
使用双词表的训练脚本

核心修改：
1. 使用 dual_vocab_builder_v2 构建两套词表
2. 使用 structure_dataset_dual_vocab 处理数据
3. 使用 structure_aware_corrector_dual_vocab 模型
"""

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import pandas as pd
from tqdm import tqdm
import numpy as np
import json
from collections import Counter


In [2]:
#版本控制
VSION="V5"

In [3]:
from structure_aware_corrector_v5 import (
    StructurePatternLearner,
    StructureAwareCorrectorV5,
    StructureAwareCorrectorDatasetV5,
    structure_aware_corrector_loss_V5,
    STRUCTURE_TYPES
)


In [4]:
from dual_vocab_builder import (
    build_dual_vocabularies,
    save_dual_vocabularies,
    load_dual_vocabularies
)

In [5]:
# ==================== 配置词表 ====================
# 加载双词表
print("\nLoading dual vocabularies...")
vocab_dir = "vocabularies_dual"
char2id, id2char, name2id, id2name = load_dual_vocabularies(vocab_dir=vocab_dir)
# ==================== 配置 ====================
class Config:
    # 数据路径
    data_path = "ppocr_resultFL_cleaned_with_noise.xlsx"
    
    # 词汇表大小
    vocab_size_ocr = len(char2id)
    vocab_size_name = len(name2id)
    
    # 模型参数
    d_model = 256
    n_heads = 8
    n_layers = 4
    dropout = 0
    
    # 训练参数
    batch_size = 32
    num_epochs = 50
    learning_rate = 1e-4
    weight_decay = 1e-5
    warmup_steps = 1000
    
    # 最长序列长度
    max_len = 128
    
    # 损失权重
    alpha_correction = 1.0
    alpha_consistency = 0.3
    
    # 保存路径
    checkpoint_dir = "./checkpoints/structure_aware_corrector_v5"
    log_dir = "./logs/structure_aware_corrector_v5"
    
    # 设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


Loading dual vocabularies...

Vocabularies loaded from 'vocabularies_dual':
  - OCR/GT vocab size: 54
  - Name vocab size: 529


In [ ]:
  
PPOCRSTR = "ppocrstr2" 


In [41]:
# ==================== 词汇表构建 ====================
def build_vocabs(df):
    """构建双词表"""
    print("构建词表...")
    
    # 提取OCR/GT和产品名称的所有字符
    ocr_chars = set()
    name_chars = set()
    
    for r in df.itertuples():
        if PPOCRSTR == "ppocrstr1":
            ocr_str = str(r.ppocrstr1)
        elif PPOCRSTR == "ppocrstr2":
            ocr_str = str(r.ppocrstr2)      
        elif PPOCRSTR == "ppocrstr3":
            ocr_str = str(r.ppocrstr3)
        gt_str = str(r.cinvstd)
        name_str = str(r.cinvname)
        
        for c in ocr_str:
            ocr_chars.add(c)
        for c in gt_str:
            ocr_chars.add(c)
        for c in name_str:
            name_chars.add(c)
    
    # 构建OCR/GT词表
    char2id = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    for i, c in enumerate(sorted(ocr_chars), start=4):
        char2id[c] = i
    id2char = {v: k for k, v in char2id.items()}
    
    # 构建产品名称词表
    name2id = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    for i, c in enumerate(sorted(name_chars), start=4):
        name2id[c] = i
    id2name = {v: k for k, v in name2id.items()}
    
    print(f"OCR词表大小: {len(char2id)}")
    print(f"名称词表大小: {len(name2id)}")
    
    return char2id, id2char, name2id, id2name

In [42]:
# ==================== 分析结构模式 ====================
def analyze_structure_patterns(df):
    """
    分析数据中的结构模式
    展示哪些产品名称具有相似的规格结构
    """
    print("\n" + "="*70)
    print("分析产品名称到规格结构的映射")
    print("="*70)
    
    # 创建结构学习器
    learner = StructurePatternLearner(max_len=64)
    
    # 从数据中学习
    learner.learn_from_data(df, 'cinvname', 'cinvstd')
    
    # 统计每个产品类别的样本数
    print(f"\n学到的结构模式数量: {len(learner.name_to_pattern)}")
    
    # 展示一些示例
    print("\n结构模式示例:")
    print("-"*70)
    
    count = 0
    for name_key, pattern in learner.name_to_pattern.items():
        if count >= 15:
            break
        
        structures = learner.name_to_structure[name_key]
        pattern_str = ''.join(pattern[:16]) + ('...' if len(pattern) > 16 else '')
        
        # 统计该模式的出现次数
        pattern_counter = Counter([''.join(s) for s in structures])
        pattern_count = pattern_counter.get(''.join(pattern), 0)
        total_samples = len(structures)
        confidence = pattern_count / total_samples if total_samples > 0 else 0
        
        # 展示一些样本
        sample_names = []
        sample_specs = []
        sample_ocrs = []
        sample_errors = []
        
        # 找到属于这个类别的样本
        sample_df = df[df['cinvname'].str.upper().str.contains(name_key[:10], na=False)].head(5)
        
        for r in sample_df.itertuples():
            sample_names.append(str(r.cinvname)[:20])
            sample_specs.append(str(r.cinvstd)[:16])
            if PPOCRSTR == "ppocrstr1": 
                sample_ocrs.append(str(r.ppocrstr1)[:16])
                sample_errors.append("✗" if str(r.ppocrstr1) != str(r.inventorycode) else "✓")
            elif PPOCRSTR == "ppocrstr2":
                sample_ocrs.append(str(r.ppocrstr2)[:16])      
                sample_errors.append("✗" if str(r.ppocrstr2) != str(r.cinvstd) else "✓")
            elif PPOCRSTR == "ppocrstr3":
                sample_ocrs.append(str(r.ppocrstr3)[:16])
                sample_errors.append("✗" if str(r.ppocrstr3) != str(r.cinvstd) else "✓")
            
        
        print(f"\n产品类别: {name_key}")
        print(f"  结构模式: {pattern_str}")
        print(f"  样本数: {total_samples} | 模式置信度: {confidence:.1%}")
        print(f"  示例:")
        print(f"    {'名称':<20} {'规格':<16} {'OCR':<16} {'正确?'}")
        print(f"    {'-'*20} {'-'*16} {'-'*16} {'-'*6}")
        for i in range(min(5, len(sample_names))):
            print(f"    {sample_names[i]:<20} {sample_specs[i]:<16} {sample_ocrs[i]:<16} {sample_errors[i]}")
        
        count += 1
    
    return learner

In [43]:
def print_evaluation_summary(metrics):
    """打印评估摘要"""
    print("\n" + "="*80)
    print("EVALUATION SUMMARY".center(80))
    print("="*80)
    print(f"Total samples: {metrics['total_samples']}")
    print(f"Corrected samples: {metrics['corrected_samples']}")
    print(f"Fix rate: {metrics['fix_rate']:.4f} ({metrics['fix_rate']*100:.2f}%)")
    print(f"Character accuracy: {metrics['char_accuracy']:.4f} ({metrics['char_accuracy']*100:.2f}%)")
    print(f"Levenshtein similarity: {metrics['levenshtein_similarity']:.4f}")
    print("\nEdit operations:")
    print(f"  Edits needed: {metrics['edits_needed']}")
    print(f"  Edits made: {metrics['edits_made']} ({metrics['edit_coverage']*100:.2f}% coverage)")
    print(f"  Correct edits: {metrics['correct_edits']}")
    print(f"  Wrong edits: {metrics['wrong_edits']}")
    print(f"  Missed edits: {metrics['edits_needed'] - metrics['correct_edits']}")
    print(f"  Correct keeps: {metrics['correct_keeps']}")
    print("\nEdit metrics:")
    print(f"  Edit recall: {metrics['edit_recall']:.4f} ({metrics['edit_recall']*100:.2f}%)")
    print(f"  Edit precision: {metrics['edit_precision']:.4f} ({metrics['edit_precision']*100:.2f}%)")
    print(f"  Structure accuracy: {metrics['structure_accuracy']:.4f} ({metrics['structure_accuracy']*100:.2f}%)")
    print("="*80)
    print(f"Fix Rate: {metrics['fix_rate']:.4f} | Edit Recall: {metrics['edit_recall']:.4f} | "
          f"Edit Precision: {metrics['edit_precision']:.4f}")
    print("="*80 + "\n")


In [44]:
# ==================== 训练函数 ====================

def train_epoch(model, dataloader, optimizer, scheduler, device, alpha_char, alpha_edit, alpha_structure):
    """训练一个 epoch"""
    model.train()

    total_loss = 0
    total_char_loss = 0
    total_edit_loss = 0
    total_structure_loss = 0

    for batch in tqdm(dataloader, desc='Training', leave=False):
        ocr_tokens = batch['ocr_tokens'].to(device)
        gt_tokens = batch['gt_tokens'].to(device)
        name_tokens = batch['name_tokens'].to(device)
        edit_targets = batch['edit_targets'].to(device)
        structure_labels = batch['structure_labels'].to(device)
        ocr_mask = batch['ocr_mask'].to(device)
        name_mask = batch['name_mask'].to(device)

        # 前向传播
        char_logits, edit_logits, structure_logits = model(
            ocr_tokens, name_tokens, ocr_mask, name_mask
        )

        # 计算损失
        loss, loss_dict = structure_aware_corrector_loss_V5(
            char_logits, gt_tokens,
            edit_logits, edit_targets,
            structure_logits, structure_labels,
            pad_id=0,
            alpha_char=alpha_char,
            alpha_edit=alpha_edit,
            alpha_structure=alpha_structure
        )

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # 更新学习率
        if scheduler:
            scheduler.step()

        # 累积损失
        total_loss += loss.item()
        total_char_loss += loss_dict['char']
        total_edit_loss += loss_dict['edit']
        total_structure_loss += loss_dict['structure']

    return {
        'total': total_loss / len(dataloader),
        'char': total_char_loss / len(dataloader),
        'edit': total_edit_loss / len(dataloader),
        'structure': total_structure_loss / len(dataloader)
    }



In [45]:
import re
# ==================== 测试结构推断 ====================
def test_structure_inference(structure_learner, df, num_samples=10):
    """测试结构推断逻辑"""
    print("\n" + "="*70)
    print("测试结构推断逻辑")
    print("="*70)
    
    count = 0
    for r in df.itertuples():
        if count >= num_samples:
            break
        if PPOCRSTR == "ppocrstr1":
            ocr = str(r.ppocrstr1).strip().upper()
        elif PPOCRSTR == "ppocrstr2":
            ocr = str(r.ppocrstr2).strip().upper()
        elif PPOCRSTR == "ppocrstr3":
            ocr = str(r.ppocrstr3).strip().upper()
        gt = str(r.cinvstd).strip().upper()
        name = str(r.cinvname).strip().upper()
        
        # 标准化
        ocr = re.sub(r'\s+', '', ocr)
        gt = re.sub(r'\s+', '', gt)
        name = re.sub(r'\s+', '', name)
        
        # 只显示有错误的样本
        if ocr == gt:
            continue
        
        print(f"\n样本 {count + 1}:")
        print(f"  产品名称: {name}")
        print(f"  OCR识别:  {ocr}")
        print(f"  正确答案: {gt}")
        print(f"  是否错误: {ocr != gt}")
        
        # 从产品名称推断结构
        pattern, confidence = structure_learner.infer_structure(name)

        # 将结构模式转换为数字ID（'A'->1, 'D'->0, 'S'->3）
        char_to_id = {'A': 1, 'D': 0, 'C': 2, 'S': 3}
        structure_pattern = [char_to_id.get(c, 4) for c in pattern]

        # 解码结构模式
        type_names = {0: '数字', 1: '字母', 2: '中文', 3: '符号', 4: '任意'}
        
        print(f"\n  推断的结构模式:")
        print(f"    置信度: {confidence:.1%}")
        
        # 分析OCR识别
        ocr_types = []
        for c in ocr:
            if re.match(r'[0-9]', c):
                ocr_types.append(0)
            elif re.match(r'[A-Z]', c):
                ocr_types.append(1)
            elif re.match(r'[\u4e00-\u9fff]', c):
                ocr_types.append(2)
            else:
                ocr_types.append(3)
        
        # 对比
        print(f"\n    位置 | 预期类型 | OCR类型 | OCR字符 | 一致?")
        print(f"    " + "-"*50)
        for i in range(min(len(ocr), 20)):
            structure_pattern = [char_to_id.get(c, 4) for c in pattern]
            expected_type = structure_pattern[i]  
            actual_type = ocr_types[i] if i < len(ocr_types) else 4
            
            expected_name = type_names[expected_type]
            actual_name = type_names[actual_type]
            
            is_consistent = (expected_type == actual_type) or (expected_type == 4)
            marker = "✓" if is_consistent else "✗"
            
            print(f"    {i:3d}  | {expected_name:8s} | {actual_name:8s} | {ocr[i]:8s} | {marker}")
        
        count += 1

In [46]:
# ==================== 训练函数 ====================
def train_epoch(model, loader, optimizer, scheduler, config):
    """训练一个epoch"""
    model.train()
    total_loss = 0
    total_corr_loss = 0
    total_cons_loss = 0
    
    pbar = tqdm(loader, desc="训练")
    for batch in pbar:
        ocr_tokens = batch['ocr_tokens'].to(config.device)
        name_tokens = batch['name_tokens'].to(config.device)
        structure_pattern = batch['structure_pattern'].to(config.device)
        target_tokens = batch['target_tokens'].to(config.device)
        consistency_mask = batch['consistency_mask'].to(config.device)
        
        # 创建mask
        ocr_mask = (ocr_tokens == 0)
        name_mask = (name_tokens == 0)
        
        # 前向传播
        correction_logits, consistency_logits = model(
            ocr_tokens, name_tokens, structure_pattern,
            ocr_mask, name_mask
        )
        
        # 计算损失
        loss, loss_dict = structure_aware_corrector_loss_V5(
            correction_logits, target_tokens,
            consistency_logits, consistency_mask,
            pad_id=0,
            alpha_correction=config.alpha_correction,
            alpha_consistency=config.alpha_consistency
        )
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        # 统计
        total_loss += loss.item()
        total_corr_loss += loss_dict['correction_loss']
        total_cons_loss += loss_dict['consistency_loss']
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'corr': f"{loss_dict['correction_loss']:.4f}",
            'cons': f"{loss_dict['consistency_loss']:.4f}"
        })
    
    avg_loss = total_loss / len(loader)
    avg_corr = total_corr_loss / len(loader)
    avg_cons = total_cons_loss / len(loader)
    
    return avg_loss, avg_corr, avg_cons


In [47]:
# ==================== 评估函数 ====================
def evaluate(model, loader, config):
    """评估模型"""
    model.eval()
    
    total_loss = 0
    correct = 0
    total = 0
    
    # 详细统计
    exact_match = 0
    total_samples = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="评估"):
            ocr_tokens = batch['ocr_tokens'].to(config.device)
            name_tokens = batch['name_tokens'].to(config.device)
            structure_pattern = batch['structure_pattern'].to(config.device)
            target_tokens = batch['target_tokens'].to(config.device)
            consistency_mask = batch['consistency_mask'].to(config.device)
            
            ocr_mask = (ocr_tokens == 0)
            name_mask = (name_tokens == 0)
            
            # 前向传播
            correction_logits, consistency_logits = model(
                ocr_tokens, name_tokens, structure_pattern,
                ocr_mask, name_mask
            )
            
            # 计算损失
            loss, loss_dict = structure_aware_corrector_loss_V5(
                correction_logits, target_tokens,
                consistency_logits, consistency_mask,
                pad_id=0,
                alpha_correction=config.alpha_correction,
                alpha_consistency=config.alpha_consistency
            )
            
            total_loss += loss.item()
            
            # 计算准确率
            preds = correction_logits.argmax(dim=-1)
            
            # 只计算非padding位置
            mask = (target_tokens != 0)
            correct += ((preds == target_tokens) & mask).sum().item()
            total += mask.sum().item()
            
            # 完全匹配统计
            for i in range(preds.size(0)):
                pred_seq = preds[i][mask[i]].cpu().numpy()
                target_seq = target_tokens[i][mask[i]].cpu().numpy()
                if np.array_equal(pred_seq, target_seq):
                    exact_match += 1
                total_samples += 1
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total if total > 0 else 0
    exact_match_rate = exact_match / total_samples if total_samples > 0 else 0
    
    return avg_loss, accuracy, exact_match_rate

In [48]:
import re
import os
import json
import pandas as pd

config = Config()

# 创建目录
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.log_dir, exist_ok=True)

print("="*70)
print("训练结构感知修正器 V5 - 最终版本")
print("="*70)
print(f"设备: {config.device}")
print(f"批次大小: {config.batch_size}")
print(f"Epochs: {config.num_epochs}")

# 加载数据
print("\n加载数据...")
df = pd.read_excel(config.data_path)
print(f"数据总量: {len(df)}")

# 构建词表
char2id, id2char, name2id, id2name = build_vocabs(df)

# 保存词表
vocab_data = {
    'char2id': char2id,
    'id2char': id2char,
    'name2id': name2id,
    'id2name': id2name,
    'vocab_size_ocr': len(char2id),
    'vocab_size_name': len(name2id)
}
vocab_path = os.path.join(config.checkpoint_dir, "vocab.json")
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)
print(f"词表已保存到: {vocab_path}")

# 分析结构模式
structure_learner = analyze_structure_patterns(df)

# 保存结构模式
structure_data = {
    name_key: {
        'pattern': pattern,
        'confidence': sum(1 for s in structures if ''.join(s) == ''.join(pattern)) / len(structures),
        'sample_count': len(structures)
    }
    for name_key, pattern in structure_learner.name_to_pattern.items()
    for structures in [structure_learner.name_to_structure[name_key]]
}
structure_path = os.path.join(config.log_dir, "structure_patterns.json")
with open(structure_path, 'w', encoding='utf-8') as f:
    json.dump(structure_data, f, ensure_ascii=False, indent=2)
print(f"\n结构模式已保存到: {structure_path}")

# 测试结构推断
test_structure_inference(structure_learner, df, num_samples=8)

# 创建数据集
print("\n创建数据集...")
dataset = StructureAwareCorrectorDatasetV5(
    df, char2id, name2id, structure_learner, max_len=config.max_len
)
print(f"数据集大小: {len(dataset)}")

# 划分训练集和验证集
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size,
    shuffle=False, num_workers=2, pin_memory=True
)

# 创建模型
print("\n创建模型...")
model = StructureAwareCorrectorV5(
    vocab_size_ocr=len(char2id),
    vocab_size_name=len(name2id),
    d_model=config.d_model,
    n_heads=config.n_heads,
    n_layers=config.n_layers,
    dropout=config.dropout
).to(config.device)

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

# 优化器和学习率调度器
optimizer = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay
)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=config.learning_rate,
    epochs=config.num_epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.1,
    anneal_strategy='cos'
)

# 训练循环
print("\n开始训练...")
print("="*70)

best_val_acc = 0
best_exact_match = 0
training_history = {
    'train_loss': [],
    'train_corr_loss': [],
    'train_cons_loss': [],
    'val_loss': [],
    'val_acc': [],
    'val_exact_match': []
}

for epoch in range(config.num_epochs):
    print(f"\nEpoch {epoch + 1}/{config.num_epochs}")
    
    # 训练
    train_loss, train_corr, train_cons = train_epoch(
        model, train_loader, optimizer, scheduler, config
    )
    
    # 评估
    val_loss, val_acc, val_exact_match = evaluate(model, val_loader, config)
    
    # 记录
    training_history['train_loss'].append(train_loss)
    training_history['train_corr_loss'].append(train_corr)
    training_history['train_cons_loss'].append(train_cons)
    training_history['val_loss'].append(val_loss)
    training_history['val_acc'].append(val_acc)
    training_history['val_exact_match'].append(val_exact_match)
    
    print(f"训练 - Loss: {train_loss:.4f} | Corr: {train_corr:.4f} | Cons: {train_cons:.4f}")
    print(f"验证 - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | ExactMatch: {val_exact_match:.4f}")
    
    # 保存最佳模型
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint_path = os.path.join(config.checkpoint_dir, "best_model.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_exact_match': val_exact_match,
            'config': config.__dict__
        }, checkpoint_path)
        print(f"✓ 保存最佳模型 (Val Acc: {val_acc:.4f})")
    
    # 保存最佳完全匹配模型
    if val_exact_match > best_exact_match:
        best_exact_match = val_exact_match
        checkpoint_path = os.path.join(config.checkpoint_dir, "best_exact_match_model.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_exact_match': val_exact_match,
            'config': config.__dict__
        }, checkpoint_path)
        print(f"✓ 保存最佳完全匹配模型 (ExactMatch: {val_exact_match:.4f})")
    
    # 定期保存
    # if (epoch + 1) % 5 == 0:
    #     checkpoint_path = os.path.join(config.checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pt")
    #     torch.save({
    #         'epoch': epoch,
    #         'model_state_dict': model.state_dict(),
    #         'optimizer_state_dict': optimizer.state_dict(),
    #         'val_acc': val_acc,
    #         'config': config.__dict__
    #     }, checkpoint_path)

# 保存训练历史
history_path = os.path.join(config.log_dir, "training_history.json")
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2)
print(f"\n训练历史已保存到: {history_path}")

print("\n" + "="*70)
print("训练完成!")
print(f"最佳验证准确率: {best_val_acc:.4f}")
print(f"最佳完全匹配率: {best_exact_match:.4f}")
print("="*70)


训练结构感知修正器 V5 - 最终版本
设备: cuda
批次大小: 32
Epochs: 50

加载数据...
数据总量: 8039
构建词表...
OCR词表大小: 70
名称词表大小: 566
词表已保存到: ./checkpoints/structure_aware_corrector_v5\vocab.json

分析产品名称到规格结构的映射

学到的结构模式数量: 945

结构模式示例:
----------------------------------------------------------------------

产品类别: QSTE420T
  结构模式: DSDAA
  样本数: 1 | 模式置信度: 100.0%
  示例:
    名称                   规格               OCR              正确?
    -------------------- ---------------- ---------------- ------
    QSTE420TM            2.5MM            2.5MM            ✓

产品类别: 40靠背管
  结构模式: DDDADSADDDDSA
  样本数: 11 | 模式置信度: 27.3%
  示例:
    名称                   规格               OCR              正确?
    -------------------- ---------------- ---------------- ------

产品类别: 直管
  结构模式: AADADDDDSA
  样本数: 74 | 模式置信度: 41.9%
  示例:
    名称                   规格               OCR              正确?
    -------------------- ---------------- ---------------- ------
    直管                   226A2-E6101-Z    226A2-E6101-Z    ✓
    直管                   6801

评估: 100%|██████████| 51/51 [00:19<00:00,  2.68it/s]


训练 - Loss: 3.5170 | Corr: 3.3820 | Cons: 0.4502
验证 - Loss: 2.7933 | Acc: 0.2860 | ExactMatch: 0.0000
✓ 保存最佳模型 (Val Acc: 0.2860)

Epoch 2/50


评估: 100%|██████████| 51/51 [00:19<00:00,  2.68it/s]


训练 - Loss: 1.3604 | Corr: 1.2342 | Cons: 0.4205
验证 - Loss: 0.5284 | Acc: 0.9348 | ExactMatch: 0.4453
✓ 保存最佳模型 (Val Acc: 0.9348)
✓ 保存最佳完全匹配模型 (ExactMatch: 0.4453)

Epoch 3/50


...

训练 - Loss: 0.0001 | Corr: 0.0001 | Cons: 0.0000
验证 - Loss: 0.0729 | Acc: 0.9920 | ExactMatch: 0.9409

Epoch 49/50


评估: 100%|██████████| 51/51 [00:19<00:00,  2.67it/s]


训练 - Loss: 0.0001 | Corr: 0.0001 | Cons: 0.0000
验证 - Loss: 0.0729 | Acc: 0.9920 | ExactMatch: 0.9409

Epoch 50/50


评估: 100%|██████████| 51/51 [00:18<00:00,  2.75it/s]

训练 - Loss: 0.0001 | Corr: 0.0001 | Cons: 0.0000
验证 - Loss: 0.0729 | Acc: 0.9920 | ExactMatch: 0.9409

训练历史已保存到: ./logs/structure_aware_corrector_v5\training_history.json

训练完成!
最佳验证准确率: 0.9921
最佳完全匹配率: 0.9422


### 绘图

In [1]:
import matplotlib.pyplot as plt

def set_paper_style():
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 14,
        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "legend.fontsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "lines.linewidth": 2,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "figure.figsize": (6, 4),
    })

set_paper_style()
def plot_loss(history):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure()

    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("train_figs/Training_and_Validation_Loss.pdf", bbox_inches="tight")
    plt.close()
def plot_sub_losses(history):

    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure()

    plt.plot(epochs, history["train_corr_loss"], label="Correction Loss")
    plt.plot(epochs, history["train_cons_loss"], label="Consistency Loss")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Sub-task Training Loss")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("train_figs/Sub-task_Training_Loss.pdf", bbox_inches="tight")
    plt.close()
def plot_val_accuracy(history):

    epochs = range(1, len(history["val_acc"]) + 1)

    plt.figure()

    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Validation Character Accuracy")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("train_figs/Validation_Accuracy.pdf", bbox_inches="tight")
    plt.close()
def plot_exact_match(history):

    epochs = range(1, len(history["val_exact_match"]) + 1)

    plt.figure()

    plt.plot(epochs, history["val_exact_match"], label="Exact Match")

    plt.xlabel("Epoch")
    plt.ylabel("Exact Match Rate")
    plt.title("Exact Match Performance")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig("train_figs/Exact_Match_Performance.pdf", bbox_inches="tight")
    plt.close()

def plot_all(history):

    # plot_loss(history)
    # plot_sub_losses(history)
    plot_val_accuracy(history)
    # plot_exact_match(history)

In [8]:
import json
history_path = r"C:\Users\10841\Desktop\ocr\carindustory\correctorFL\logs\structure_aware_corrector_v5\training_history.json"
with open(history_path, 'r', encoding='utf-8') as f:
    training_history = json.load(f)

print(training_history)
plot_all(training_history)

{'train_loss': [3.5170237152137567, 1.3603679834313653, 0.34693358497536597, 0.15363974261343183, 0.1060320938478655, 0.08101632145803365, 0.06591558637588624, 0.054219596766732374, 0.04592180656806672, 0.03837801985308269, 0.0326959585483691, 0.028325428953394294, 0.024325428311643537, 0.020536328854145882, 0.018234843501719226, 0.016662492128946257, 0.012971291590171555, 0.011622605547054082, 0.008776317835340637, 0.008265748470894233, 0.006110383430930243, 0.005813981935906627, 0.005361326752967243, 0.004595376009110533, 0.004745527531862472, 0.004300934598932914, 0.002296779151839107, 0.0019040804587140104, 0.0011202006230700706, 0.0007572645974587023, 0.0006834390500516385, 0.0005050535319287292, 0.00046515340175605914, 0.0003458535959260696, 0.0002921808837025093, 0.0002462139984064872, 0.0002564278125755121, 0.000256826345407287, 0.000211585660010984, 0.0001814670586778692, 0.00016560888078816083, 0.0001637713707783966, 0.00015263016225549448, 0.0001465924835420128, 0.0001409168

### 评估

In [49]:
PPOCRSTR = "ppocrstr2" 

In [50]:
def norm_str(s):
    """标准化字符串"""
    s = str(s).strip().upper()
    s = s.replace("×", "X")
    s = s.replace("（", "(").replace("）", ")")
    s = re.sub(r"\s+", "", s)
    return s

In [51]:
def levenshtein_distance(s1, s2):
    """计算编辑距离"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

In [52]:
from difflib import SequenceMatcher
def calculate_similarity(s1, s2):
    """计算字符串相似度"""
    return SequenceMatcher(None, s1, s2).ratio()

def analyze_error_type(ocr, gt):
    """
    分析错误类型
    
    Returns:
        dict: {
            'error_type': 'correct'|'insert'|'delete'|'substitute'|'mixed',
            'num_errors': int,
            'length_diff': int
        }
    """
    if ocr == gt:
        return {'error_type': 'correct', 'num_errors': 0, 'length_diff': 0}
    
    # 计算编辑距离
    distance = levenshtein_distance(ocr, gt)
    length_diff = len(ocr) - len(gt)
    
    # 简单分类
    if distance == 1:
        if length_diff == 1:
            error_type = 'insert'
        elif length_diff == -1:
            error_type = 'delete'
        else:
            error_type = 'substitute'
    else:
        error_type = 'mixed'
    
    return {
        'error_type': error_type,
        'num_errors': distance,
        'length_diff': length_diff
    }


In [66]:
def classify_difficulty(ocr, gt):
    """
    根据编辑距离分类难度
    
    Returns:
        'easy': 0-1个错误
        'medium': 2-3个错误
        'hard': 4个及以上错误
    """
    distance = levenshtein_distance(ocr, gt)
    if distance == 0:
        return 'correct'
    elif distance <= 1:
        return 'easy'
    elif distance <= 3 and distance > 1:
        return 'medium'
    else:
        return 'hard'

In [ ]:
from tqdm import tqdm
from collections import Counter, defaultdict

# ==================== 评估类 ====================
class V5ModelEvaluator:
    """V5模型评估器"""
    
    def __init__(self, model_path, vocab_path, structure_path, device='cpu'):
        """
        Args:
            model_path: 模型检查点路径
            vocab_path: 词表路径
            structure_path: 结构模式路径
            device: 设备
        """
        self.device = device
        
        # 加载词表
        print(f"加载词表: {vocab_path}")
        with open(vocab_path, 'r', encoding='utf-8') as f:
            vocab_data = json.load(f)
        self.char2id = vocab_data['char2id']
        self.id2char = {int(k): v for k, v in vocab_data['id2char'].items()}
        self.name2id = vocab_data['name2id']
        self.id2name = {int(k): v for k, v in vocab_data['id2name'].items()}
        
        # 加载结构模式
        print(f"加载结构模式: {structure_path}")
        with open(structure_path, 'r', encoding='utf-8') as f:
            structure_data = json.load(f)
        self.structure_learner = StructurePatternLearner()
        self.structure_learner.name_to_pattern = {
            k: v['pattern']
            for k, v in structure_data.items()
        }
        
        # 加载模型
        print(f"加载模型: {model_path}")
        checkpoint = torch.load(model_path, map_location=device)
        
        # 创建模型
        self.model = StructureAwareCorrectorV5(
            vocab_size_ocr=vocab_data['vocab_size_ocr'],
            vocab_size_name=vocab_data['vocab_size_name'],
            d_model=256,
            n_heads=8,
            n_layers=4,
            dropout=0.2
        ).to(device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.eval()
        
        print("模型加载完成!")
    
    def encode(self, s, char2id, max_len=128):
        """编码字符串"""
        s = norm_str(s)
        ids = [char2id["<SOS>"]]
        unk = char2id.get("<UNK>", char2id["<PAD>"])
        
        for c in s:
            ids.append(char2id.get(c, unk))
        
        ids.append(char2id["<EOS>"])
        
        # 填充
        if len(ids) > max_len:
            ids = ids[:max_len]
        else:
            ids = ids + [char2id["<PAD>"]] * (max_len - len(ids))
        
        return ids
    
    def infer_structure(self, name):
        """推断结构模式"""
        name = norm_str(name)
        name_key = self.structure_learner.get_name_key(name)
        
        if name_key in self.structure_learner.name_to_pattern:
            pattern = self.structure_learner.name_to_pattern[name_key]
        else:
            pattern = []
        
        # 转换为数字ID
        char_to_id = {'A': 1, 'D': 0, 'C': 2, 'S': 3}
        structure_pattern = [char_to_id.get(c, 4) for c in pattern]
        
        # 填充
        max_len = 128
        if len(structure_pattern) < max_len:
            structure_pattern = structure_pattern + [4] * (max_len - len(structure_pattern))
        else:
            structure_pattern = structure_pattern[:max_len]
        
        return structure_pattern
    
    @torch.no_grad()
    def predict(self, ocr_str, name_str):
        """
        预测修正结果
        
        Returns:
            corrected_str: 修正后的字符串
        """
        ocr = norm_str(ocr_str)
        name = norm_str(name_str)
        
        # 编码
        ocr_tokens = torch.LongTensor([self.encode(ocr, self.char2id)]).to(self.device)
        name_tokens = torch.LongTensor([self.encode(name, self.name2id)]).to(self.device)
        structure_pattern = torch.LongTensor([self.infer_structure(name)]).to(self.device)
        
        # 创建mask
        ocr_mask = (ocr_tokens == 0)
        name_mask = (name_tokens == 0)
        
        # 前向传播
        correction_logits, _ = self.model(
            ocr_tokens, name_tokens, structure_pattern,
            ocr_mask, name_mask
        )
        
        # 解码
        preds = correction_logits.argmax(dim=-1)[0]
        
        # 移除padding和EOS
        corrected_ids = []
        for pid in preds:
            if pid == 0:  # PAD
                continue
            if pid == 2:  # EOS
                break
            if pid not in [1, 2]:  # SOS, EOS
                corrected_ids.append(pid.item())
        
        # 转换为字符串
        corrected_str = ''.join([self.id2char[pid] for pid in corrected_ids])
        
        return corrected_str
    
    def evaluate_dataset(self, df, ocr_col=PPOCRSTR, gt_col='cinvstd', name_col='cinvname'):
        """
        评估整个数据集
        
        Returns:
            dict: 详细的评估结果
        """
        print("\n开始评估...")
        print("="*70)
        
        results = {
            'char_level': defaultdict(list),
            'string_level': defaultdict(list),
            'error_analysis': defaultdict(list),
            'difficulty_analysis': defaultdict(list),
            'predictions': []
        }
        
        for r in tqdm(df.itertuples(), total=len(df), desc="评估中"):
            ocr = norm_str(getattr(r, ocr_col))
            gt = norm_str(getattr(r, gt_col))
            name = norm_str(getattr(r, name_col))
            
            # 初步提取结果
            ocr_error = analyze_error_type(ocr, gt)
            
            # V5模型预测
            v5_pred = self.predict(ocr, name)
            v5_error = analyze_error_type(v5_pred, gt)
            
            # 字符级别指标
            # 计算字符级别的准确率
            min_len = min(len(gt), len(v5_pred))
            if min_len > 0:
                char_correct = sum(1 for i in range(min_len) if v5_pred[i] == gt[i])
                results['char_level']['char_accuracy'].append(char_correct / min_len)
            
            # 初步提取的字符准确率
            min_len_ocr = min(len(gt), len(ocr))
            if min_len_ocr > 0:
                ocr_char_correct = sum(1 for i in range(min_len_ocr) if ocr[i] == gt[i])
                results['char_level']['ocr_char_accuracy'].append(ocr_char_correct / min_len_ocr)
            
            # 字符串级别指标
            results['string_level']['v5_exact_match'].append(v5_pred == gt)
            results['string_level']['ocr_exact_match'].append(ocr == gt)
            
            # 编辑距离
            v5_distance = levenshtein_distance(v5_pred, gt)
            ocr_distance = levenshtein_distance(ocr, gt)
            
            results['string_level']['v5_edit_distance'].append(v5_distance)
            results['string_level']['ocr_edit_distance'].append(ocr_distance)
            
            # 相似度
            results['string_level']['v5_similarity'].append(calculate_similarity(v5_pred, gt))
            results['string_level']['ocr_similarity'].append(calculate_similarity(ocr, gt))
            
            # 错误类型分析
            results['error_analysis']['v5_error_type'].append(v5_error['error_type'])
            results['error_analysis']['ocr_error_type'].append(ocr_error['error_type'])
            
            # 难度级别分析
            difficulty = classify_difficulty(ocr, gt)
            results['difficulty_analysis']['difficulty'].append(difficulty)
            results['difficulty_analysis'][f'v5_correct_{difficulty}'].append(v5_pred == gt)
            results['difficulty_analysis'][f'ocr_correct_{difficulty}'].append(ocr == gt)
            
            # 保存预测结果
            results['predictions'].append({
                'ocr': ocr,
                'v5_pred': v5_pred,
                'gt': gt,
                'name': name,
                'ocr_error_type': ocr_error['error_type'],
                'v5_error_type': v5_error['error_type'],
                'ocr_correct': ocr == gt,
                'v5_correct': v5_pred == gt,
                'difficulty': difficulty
            })
        
        return results
    
    def print_evaluation_report(self, results):
        """打印评估报告"""
        print("\n" + "="*70)
        print("V4模型性能评估报告")
        print("="*70)
        
        # 1. 字符级别指标
        print("\n【字符级别指标】")
        print("-"*70)
        
        ocr_char_acc = np.mean(results['char_level']['ocr_char_accuracy'])
        v5_char_acc = np.mean(results['char_level']['char_accuracy'])
        improvement_char = v5_char_acc - ocr_char_acc
        improvement_char_pct = (improvement_char / ocr_char_acc) * 100 if ocr_char_acc > 0 else 0
        
        print(f"字符准确率:")
        print(f"  初步提取 ({PPOCRSTR}): {ocr_char_acc:.4f} ({ocr_char_acc*100:.2f}%)")
        print(f"  V5模型修正:         {v5_char_acc:.4f} ({v5_char_acc*100:.2f}%)")
        print(f"  提升幅度:           {improvement_char:+.4f} ({improvement_char_pct:+.2f}%)")
        
        # 2. 字符串级别指标
        print("\n【字符串级别指标】")
        print("-"*70)
        
        ocr_exact = np.mean(results['string_level']['ocr_exact_match'])
        v5_exact = np.mean(results['string_level']['v5_exact_match'])
        improvement_exact = v5_exact - ocr_exact
        improvement_exact_pct = (improvement_exact / ocr_exact) * 100 if ocr_exact > 0 else 0
        
        print(f"完全匹配率:")
        print(f"  初步提取 ({PPOCRSTR}): {ocr_exact:.4f} ({ocr_exact*100:.2f}%)")
        print(f"  V5模型修正:         {v5_exact:.4f} ({v5_exact*100:.2f}%)")
        print(f"  提升幅度:           {improvement_exact:+.4f} ({improvement_exact_pct:+.2f}%)")
        
        # 编辑距离
        ocr_avg_distance = np.mean(results['string_level']['ocr_edit_distance'])
        v5_avg_distance = np.mean(results['string_level']['v5_edit_distance'])
        improvement_dist = ocr_avg_distance - v5_avg_distance
        
        print(f"\n平均编辑距离:")
        print(f"  初步提取 ({PPOCRSTR}): {ocr_avg_distance:.4f}")
        print(f"  V5模型修正:         {v5_avg_distance:.4f}")
        print(f"  减少:               {improvement_dist:.4f} ({improvement_dist/ocr_avg_distance*100:.2f}%)")
        
        # 相似度
        ocr_sim = np.mean(results['string_level']['ocr_similarity'])
        v5_sim = np.mean(results['string_level']['v5_similarity'])
        improvement_sim = v5_sim - ocr_sim
        
        print(f"\n平均相似度:")
        print(f"  初步提取 ({PPOCRSTR}): {ocr_sim:.4f} ({ocr_sim*100:.2f}%)")
        print(f"  V5模型修正:         {v5_sim:.4f} ({v5_sim*100:.2f}%)")
        print(f"  提升:               {improvement_sim:+.4f} ({improvement_sim*100:+.2f}%)")
        
        # 3. 错误类型分析
        print("\n【错误类型分析】")
        print("-"*70)
        
        ocr_error_types = Counter(results['error_analysis']['ocr_error_type'])
        v5_error_types = Counter(results['error_analysis']['v5_error_type'])
        
        print("\n初步提取 (PPOCRSTR) 错误类型分布:")
        for error_type in ['correct', 'substitute', 'insert', 'delete', 'mixed']:
            count = ocr_error_types.get(error_type, 0)
            pct = count / len(results['predictions']) * 100
            print(f"  {error_type:12s}: {count:6d} ({pct:5.2f}%)")
        
        print("\nV5模型修正后错误类型分布:")
        for error_type in ['correct', 'substitute', 'insert', 'delete', 'mixed']:
            count = v5_error_types.get(error_type, 0)
            pct = count / len(results['predictions']) * 100
            print(f"  {error_type:12s}: {count:6d} ({pct:5.2f}%)")
        
        # 4. 难度级别分析
        print("\n【难度级别分析】")
        print("-"*70)
        print(f"根据编辑距离将样本分为三类难度：")
        print(f"  - Easy: 1个错误:{len(results['difficulty_analysis']['v5_correct_easy'])}个样本")
        print(f"  - Medium: 2-3个错误:{len(results['difficulty_analysis']['v5_correct_medium'])}个样本")
        print(f"  - Hard: 4个及以上错误:{len(results['difficulty_analysis']['v5_correct_hard'])}个样本")   

        
        for difficulty in ['easy', 'medium', 'hard']:
            ocr_correct = np.mean(results['difficulty_analysis'][f'ocr_correct_{difficulty}'])
            v5_correct = np.mean(results['difficulty_analysis'][f'v5_correct_{difficulty}'])
            improvement = v5_correct - ocr_correct
            
            print(f"\n{difficulty.upper()} 难度样本:")
            print(f"  初步提取准确率: {ocr_correct:.4f} ({ocr_correct*100:.2f}%)")
            print(f"  V5模型准确率:   {v5_correct:.4f} ({v5_correct*100:.2f}%)")
            print(f"  提升:           {improvement:+.4f} ({improvement*100:+.2f}%)")
        
        # 5. 总结
        print("\n" + "="*70)
        print("【总结】")
        print("="*70)
        print(f"总样本数: {len(results['predictions'])}")
        print(f"\n关键提升:")
        print(f"  字符准确率提升: {improvement_char:+.4f} ({improvement_char_pct:+.2f}%)")
        print(f"  完全匹配率提升: {improvement_exact:+.4f} ({improvement_exact_pct:+.2f}%)")
        print(f"  编辑距离减少:   {improvement_dist:.4f} ({improvement_dist/ocr_avg_distance*100:.2f}%)")
        print(f"  相似度提升:     {improvement_sim:+.4f} ({improvement_sim*100:+.2f}%)")
        print("="*70)


In [69]:
import re
"""
在Jupyter Notebook中运行的简化评估函数

Args:
    config: 配置字典
    sample_size: 如果指定，只评估前N个样本（用于快速测试）

Returns:
    results: 评估结果字典
    summary: 关键指标摘要
"""
config = {
    'data_path': './ppocr_resultFL_cleaned_with_noise.xlsx',
    'model_path': './checkpoints/structure_aware_corrector_v5/best_model.pt',
    'vocab_path': './checkpoints/structure_aware_corrector_v5/vocab.json',
    'structure_path': './logs/structure_aware_corrector_v5/structure_patterns.json',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# 加载数据
print("\n加载数据...")
df = pd.read_excel("ppocr_resultFL_cleaned_with_noise.xlsx")


# 划分验证集
from sklearn.model_selection import train_test_split
_, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"验证集大小: {len(val_df)}")

# 创建评估器
evaluator = V5ModelEvaluator(
    model_path=config['model_path'],
    vocab_path=config['vocab_path'],
    structure_path=config['structure_path'],
    device=config['device']
)

# 评估
results = evaluator.evaluate_dataset(val_df)

# 打印报告
summary = evaluator.print_evaluation_report(results)

# 保存结果
output_path = './logs/structure_aware_corrector_v5/evaluation_results.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    results_serializable = {}
    for key, value in results.items():
        if key == 'predictions':
            results_serializable[key] = value
        else:
            results_serializable[key] = {k: [float(v) if isinstance(v, (np.float64, np.int64)) else v for v in val] 
                                            for k, val in value.items()}
    json.dump(results_serializable, f, ensure_ascii=False, indent=2)
print(f"\n详细结果已保存到: {output_path}")

# 保存预测结果
predictions_df = pd.DataFrame(results['predictions'])
predictions_path = './logs/structure_aware_corrector_v5/predictions.csv'
predictions_df.to_csv(predictions_path, index=False, encoding='utf-8-sig')
print(f"预测结果已保存到: {predictions_path}")



加载数据...
验证集大小: 1608
加载词表: ./checkpoints/structure_aware_corrector_v5/vocab.json
加载结构模式: ./logs/structure_aware_corrector_v5/structure_patterns.json
加载模型: ./checkpoints/structure_aware_corrector_v5/best_model.pt


C:\Users\10841\AppData\Local\Temp\ipykernel_15652\1862635577.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


模型加载完成!

开始评估...


评估中: 100%|██████████| 1608/1608 [00:29<00:00, 54.43it/s]



V4模型性能评估报告

【字符级别指标】
----------------------------------------------------------------------
字符准确率:
  初步提取 (ppocrstr2): 0.9723 (97.23%)
  V5模型修正:         0.9963 (99.63%)
  提升幅度:           +0.0240 (+2.47%)

【字符串级别指标】
----------------------------------------------------------------------
完全匹配率:
  初步提取 (ppocrstr2): 0.8414 (84.14%)
  V5模型修正:         0.9658 (96.58%)
  提升幅度:           +0.1244 (+14.78%)

平均编辑距离:
  初步提取 (ppocrstr2): 0.2755
  V5模型修正:         0.5995
  减少:               -0.3240 (-117.61%)

平均相似度:
  初步提取 (ppocrstr2): 0.9754 (97.54%)
  V5模型修正:         0.9931 (99.31%)
  提升:               +0.0177 (+1.77%)

【错误类型分析】
----------------------------------------------------------------------

初步提取 (PPOCRSTR) 错误类型分布:
  correct     :   1353 (84.14%)
  substitute  :    138 ( 8.58%)
  insert      :      1 ( 0.06%)
  delete      :      8 ( 0.50%)
  mixed       :    108 ( 6.72%)

V5模型修正后错误类型分布:
  correct     :   1553 (96.58%)
  substitute  :     22 ( 1.37%)
  insert      :      0 ( 0.00%)
  delet

In [70]:
"""
规格型号字符串长度分布直方图绘制
符合计算机顶会风格（CVPR/ICCV/NeurIPS等）
"""

import pandas as pd
import matplotlib.pyplot as plt


def set_top_conference_style():
    """设置计算机顶会风格的绘图参数"""
    plt.rcParams.update({
        # 字体设置
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'font.size': 11,
        'axes.labelsize': 12,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        
        # 线条和标记
        'lines.linewidth': 1.5,
        'lines.markersize': 6,
        'patch.linewidth': 0.5,
        
        # 图形质量
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        
        # 网格
        'axes.grid': True,
        'grid.alpha': 0.2,
        'grid.linestyle': '--',
        'grid.linewidth': 0.5,
        
        # 其他
        'axes.unicode_minus': False,
        'text.usetex': False,
    })


def plot_length_histogram(xlsx_file, column='cinvstd', save_path='length_histogram.png'):
    """
    绘制规格型号字符串长度分布直方图
    
    参数:
        xlsx_file: Excel文件路径
        column: 规格型号字段名称（默认为'cinvstd'）
        save_path: 图片保存路径
    """
    # 设置顶会风格
    set_top_conference_style()
    
    # 读取Excel文件
    df = pd.read_excel(xlsx_file)
    
    # 提取字符串并计算长度
    strings = df[column].dropna().astype(str)
    lengths = strings.str.len()
    
    # 计算统计信息
    count = len(strings)
    mean_val = lengths.mean()
    std_val = lengths.std()
    median_val = lengths.median()
    min_val = int(lengths.min())
    max_val = int(lengths.max())
    
    # 自动确定分箱数量
    q25 = lengths.quantile(0.25)
    q75 = lengths.quantile(0.75)
    iqr = q75 - q25
    if iqr > 0:
        bin_width = 2 * iqr / (count ** (1/3))
        bins = max(1, int((max_val - min_val) / bin_width))
    else:
        bins = min(30, len(set(lengths)))
    
    # 创建图形
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # 绘制直方图
    n, bins_edges, patches = ax.hist(
        lengths, 
        bins=bins, 
        alpha=0.7, 
        color='#4472C4',
        edgecolor='white',
        linewidth=0.5
    )
    
    # 添加统计线
    ax.axvline(mean_val, color='red', linestyle='--', 
               linewidth=1.5, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='-.', 
               linewidth=1.5, label=f'Median: {median_val:.1f}')
    
    # 设置标签和标题
    ax.set_xlabel('String Length', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count', fontsize=12, fontweight='bold')
    ax.set_title('Distribution of Specification String Lengths', 
                 fontsize=12, fontweight='bold', pad=10)
    
    # 设置坐标轴范围
    ax.set_xlim(left=0, right=max(max_val * 1.05, mean_val + 3 * std_val))
    
    # 添加统计信息文本框（左上角，避免与图例重叠）
    textstr = f'''Statistics:
Count: {count:,}
Mean: {mean_val:.2f} ± {std_val:.2f}
Median: {median_val:.2f}
Min: {min_val}, Max: {max_val}'''
    
    props = dict(boxstyle='round', facecolor='wheat', alpha=0.3)
    ax.text(0.03, 0.97, textstr, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', horizontalalignment='left', 
            bbox=props, fontfamily='monospace')
    
    # 图例（中上方，水平居中，2列布局）
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.0), 
              ncol=2, framealpha=0.9, edgecolor='gray')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"直方图已保存到: {save_path}")
    plt.close()
    
    # 打印统计信息
    print("\n" + "="*50)
    print("规格型号字符串长度统计")
    print("="*50)
    print(f"总样本数: {count:,}")
    print(f"均值: {mean_val:.2f}")
    print(f"标准差: {std_val:.2f}")
    print(f"中位数: {median_val:.2f}")
    print(f"最小值: {min_val}")
    print(f"最大值: {max_val}")
    print("="*50 + "\n")


if __name__ == "__main__":
    # 使用示例
    xlsx_file = 'ppocr_resultFL_cleaned_with_noise.xlsx'  # 替换为你的Excel文件路径
    
    # 绘制直方图
    plot_length_histogram(
        xlsx_file=xlsx_file,
        column='cinvstd',  # 规格型号字段名称
        save_path='length_histogram.pdf'
    )


直方图已保存到: length_histogram.pdf

规格型号字符串长度统计
总样本数: 8,039
均值: 12.16
标准差: 3.82
中位数: 12.00
最小值: 2
最大值: 50

